# MeetSense AI - Meeting Intelligence Engine

## Objective

Generate an AI-powered meeting report including:

- Executive Summary
- Key Decisions
- Action Items
- Deadlines
- Responsible Members
- Risks
- AI Recommendations

In [2]:
from transformers import pipeline
import os
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

In [3]:
transcript_path = "../outputs/meeting_transcript.txt"

with open(transcript_path, "r", encoding="utf-8") as file:
    transcript = file.read()

print(transcript)

 Good morning everyone, thank you for joining today's project planning meeting. Our objective is to finalize the timeline for the AI meeting and diligent system and assigned responsibilities for the first development sprint. Good morning, I have completed the initial research on the speech recognition models. I recommend using open AI whisper because it provides accurate transcription and works well even with the background noise. That sounds good. I will start working on the backend APIs that will process the transcript and communicate with the AI models. I expect to finish the API development by next Monday. From the design perspective, I have prepared a few dashboard wireframes. The dashboard will display the meeting transcript, AI generated summary, action items, sentiment analysis and productivity score in a clean and user friendly layout. Excellent, we also need to make sure that the streamlit application is responsive and easy to use. Please keep the interface simple because our

## Load FLAN-T5 Model

In [4]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("FLAN-T5 Loaded Successfully!")

Device set to use cpu


FLAN-T5 Loaded Successfully!


In [5]:
## Generate Executive Summary
summary_prompt = f"""
You are an AI meeting assistant.

Generate a professional executive summary of the following meeting in 5 concise bullet points.

Meeting Transcript:
{transcript}
"""

summary = generator(
    summary_prompt,
    max_new_tokens=200,
    do_sample=False
)

executive_summary = summary[0]["generated_text"]

print(executive_summary)

Good morning everyone, thank you for joining today's project planning meeting. Our objective is to finalize the timeline for the AI meeting and diligent system and assigned responsibilities for the first development sprint. Good morning, I have completed the initial research on the speech recognition models. I recommend using open AI whisper because it provides accurate transcription and works well even with the background noise. I will start working on the backend APIs that will process the transcript and communicate with the AI models. I have prepared a few dashboard wireframes. The dashboard will display the meeting transcript, AI generated summary, action items, sentiment analysis and productivity score in a clean and user friendly layout. I will also need to make sure that the streamlit application is responsive and easy to use. Our deadline for completing the first prototype is next Friday before that everyone should push their latest code to the GitHub and update the project doc

In [6]:
# Extract Key Decisions
decision_prompt = f"""
You are an AI meeting assistant.

Extract only the important decisions made during the meeting.

Return them as bullet points.

Meeting Transcript:
{transcript}
"""

decision = generator(
    decision_prompt,
    max_new_tokens=180,
    do_sample=False
)

key_decisions = decision[0]["generated_text"]

print(key_decisions)

Meeting Transcript: Good morning everyone, thank you for joining today's project planning meeting. Our objective is to finalize the timeline for the AI meeting and diligent system and assigned responsibilities for the first development sprint. Good morning, I have completed the initial research on the speech recognition models. I recommend using open AI whisper because it provides accurate transcription and works well even with the background noise. That sounds good. I will start working on the backend APIs that will process the transcript and communicate with the AI models. I have prepared a few dashboard wireframes. The dashboard will display the meeting transcript, AI generated summary, action items, sentiment analysis and productivity score in a clean and user friendly layout. I will also need to make sure that the streamlit application is responsive and easy to use. Our deadline for completing the first prototype is next Friday before that everyone should push


In [8]:
## Extract tasks and deadlines 
import re
import pandas as pd

# Split transcript into sentences
sentences = re.split(r'(?<=[.!?])\s+', transcript)

# Action words
action_keywords = [
    "will",
    "should",
    "need to",
    "complete",
    "finish",
    "prepare",
    "start",
    "integrate",
    "develop",
    "finalize",
    "perform",
    "update",
    "push",
    "submit",
    "review",
    "test"
]

# Deadline patterns
deadline_patterns = [
    r"by\s+[A-Za-z]+",
    r"next\s+[A-Za-z]+",
    r"this\s+[A-Za-z]+",
    r"before\s+[A-Za-z]+"
]

action_items = []

for sentence in sentences:

    text = sentence.strip()

    if any(keyword in text.lower() for keyword in action_keywords):

        deadline = "Not Mentioned"

        for pattern in deadline_patterns:

            match = re.search(pattern, text.lower())

            if match:
                deadline = match.group().title()
                break

        action_items.append({
            "Task": text,
            "Deadline": deadline
        })

df_actions = pd.DataFrame(action_items)

df_actions

,Task,Deadline
0,Our objective is to finalize the timeline for ...,Not Mentioned
1,"Good morning, I have completed the initial res...",Not Mentioned
2,I will start working on the backend APIs that ...,Not Mentioned
3,I expect to finish the API development by next...,By Next
4,"From the design perspective, I have prepared a...",Not Mentioned
5,The dashboard will display the meeting transcr...,Not Mentioned
6,"Excellent, we also need to make sure that the ...",Not Mentioned
7,This feature will help user quickly understand...,This Feature
8,We should also include an action item extracto...,Not Mentioned
9,Our deadline for completing the first prototyp...,Next Friday


In [10]:
## Identify Potential Risks
risk_keywords = [
    "delay",
    "deadline",
    "risk",
    "challenge",
    "issue",
    "problem",
    "testing",
    "bug",
    "dependency",
    "client",
    "documentation"
]

risks = []

for sentence in sentences:

    text = sentence.strip()

    if any(word in text.lower() for word in risk_keywords):
        risks.append(text)

if len(risks) == 0:
    risks.append(
        "No major risks were explicitly discussed. Continue monitoring project deadlines and testing progress."
    )

meeting_risks = "\n".join(f"• {risk}" for risk in risks)

print(meeting_risks)

• We should also include an action item extractor that automatically identifies assigned task, responsible team members and deadlines that will improve productivity and reduce manual note-taking.
• Our deadline for completing the first prototype is next Friday before that everyone should push their latest code to the GitHub and update the project documentation.
• I will also perform testing on different meeting records to verify the transcription accuracy and ensure the summaries are meaningful.
• If everything goes according to plan, we should be ready for the client demonstration next Tuesday.


In [11]:
recommendations = []

if len(df_actions) > 0:
    recommendations.append("Track all action items until completion.")

if "testing" in transcript.lower():
    recommendations.append("Perform regular testing before deployment.")

if "deadline" in transcript.lower() or "friday" in transcript.lower():
    recommendations.append("Monitor project deadlines carefully.")

recommendations.append("Schedule weekly progress review meetings.")
recommendations.append("Maintain project documentation on GitHub.")

ai_recommendations = "\n".join(f"• {rec}" for rec in recommendations)

print(ai_recommendations)

• Track all action items until completion.
• Perform regular testing before deployment.
• Monitor project deadlines carefully.
• Schedule weekly progress review meetings.
• Maintain project documentation on GitHub.
